# 🤖 Building a Chatbot with Message History using LangChain

In this notebook we build an **LLM-powered chatbot** step by step and learn how to make it *remember* the conversation.

**What we cover:**
1. Loading API keys from a `.env` file
2. Connecting to an LLM (Groq) using `ChatGroq`
3. Sending messages to the model (`HumanMessage`, `AIMessage`)
4. Adding **Message History** so the bot remembers past messages (`RunnableWithMessageHistory`)
5. Using **Prompt Templates** (`ChatPromptTemplate`, `MessagesPlaceholder`)
6. Adding extra input variables to the prompt (e.g. `language`)
7. Combining prompt templates with message history when there are multiple input keys
8. **Trimming** the conversation history so it fits the model's context window (`trim_messages`)

> 💡 **Key idea:** LLMs are *stateless* — they don't remember anything between calls. To build a chatbot, we must send the previous conversation along with every new message. LangChain's message history tools do this for us automatically.

## 1. Load Environment Variables

We keep secret keys (like `GROQ_API_KEY`) inside a `.env` file instead of writing them directly in the code.
`load_dotenv()` reads that file and loads the keys as environment variables, so we can access them with `os.getenv()`.

In [10]:
## Building the chatbot (LLM Powered Chatbot)
import os                       # to read environment variables
from dotenv import load_dotenv  # to load variables from the .env file

load_dotenv()  # reads .env and loads keys into the environment (returns True if successful)

True

In [11]:
# Read the Groq API key from the environment variables
groq_api_key = os.getenv("GROQ_API_KEY")

## 2. Initialize the LLM (Groq)

`ChatGroq` is LangChain's wrapper for models hosted on **Groq** (very fast inference).
We create a `model` object that we will use throughout the notebook.

In [12]:
import os
from langchain_groq import ChatGroq  # LangChain integration for Groq-hosted models

# Create the chat model
model = ChatGroq(
    model="openai/gpt-oss-20b",              # name of the model hosted on Groq
    groq_api_key=os.getenv("GROQ_API_KEY")   # API key used to authenticate
)

## 3. Talking to the Model with Messages

Chat models take a **list of messages** as input. LangChain provides message classes for each role:

| Class | Role | Meaning |
|---|---|---|
| `SystemMessage` | system | Instructions that set the assistant's behaviour |
| `HumanMessage` | user | What the user says |
| `AIMessage` | assistant | What the model replied |

`model.invoke(...)` sends the messages and returns an `AIMessage` with the reply.

In [13]:
from langchain_core.messages import HumanMessage

# Send a single user message to the model
model.invoke([
    HumanMessage(
        content = " Hi my name is yoki and I am an AI engineer"
        )
    ])

AIMessage(content='Hello, Yoki! 👋 It’s great to meet an AI engineer. What projects are you currently working on? Whether it’s model fine‑tuning, data pipelines, or something else, I’m happy to chat, help troubleshoot, or dive into any AI topic you’re interested in. Let me know what’s on your mind!', additional_kwargs={'reasoning_content': 'We need to respond. The user says "Hi my name is yoki and I am an AI engineer". We should respond warmly. Maybe ask about their interests, or what they are working on. Provide helpful info.'}, response_metadata={'token_usage': {'completion_tokens': 122, 'prompt_tokens': 82, 'total_tokens': 204, 'completion_time': 0.128495328, 'completion_tokens_details': {'reasoning_tokens': 44}, 'prompt_time': 0.00402464, 'prompt_tokens_details': None, 'queue_time': 0.376229638, 'total_time': 0.132519968}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_aef561cf8e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider

### Manually passing the conversation history

Because the model is **stateless**, if we ask *"what is my name?"* in a new call, it won't know.
To make it remember, we pass the **whole conversation** — previous user messages (`HumanMessage`) **and** previous model replies (`AIMessage`) — together with the new question.

In [ ]:
from langchain_core.messages import AIMessage

# Pass the full conversation so the model has context
model.invoke([
    # 1st turn: user introduces themselves
    HumanMessage(
        content = "Hi my name is yoki and I am an AI engineer"
    ),
    # Model's previous reply (we add it manually here)
    AIMessage(
        content = "Hello Yoki! 👋 It’s great to meet an AI engineer. What kind of projects are you working on right now? Whether you’re building models, optimizing pipelines, or exploring new research, I’d love to hear more about it."
    ),
    # 2nd turn: new question that depends on earlier context
    HumanMessage(
        content = "Hey what is my name and what do i do?"
    )
])
# ✅ The model answers correctly because the earlier messages were included

AIMessage(content='You’re Yoki, and you’re an AI engineer. 🚀 If you’d like to dive into any specific projects or talk about the work you’re doing, just let me know!', additional_kwargs={'reasoning_content': 'User: "Hey what is my name and waht do i do?" They want to confirm their name and what they do. We know from earlier that their name is Yoki and they are an AI engineer. The user might be testing. We should answer politely, acknowledging their name and role. Also correct spelling of "waht" to "what." Provide answer: "Your name is Yoki, and you are an AI engineer." Possibly ask if they want more detail. Let\'s respond accordingly.'}, response_metadata={'token_usage': {'completion_tokens': 149, 'prompt_tokens': 153, 'total_tokens': 302, 'completion_time': 0.156876726, 'completion_tokens_details': {'reasoning_tokens': 102}, 'prompt_time': 0.010540173, 'prompt_tokens_details': None, 'queue_time': 0.153923916, 'total_time': 0.167416899}, 'model_name': 'openai/gpt-oss-20b', 'system_finge

## 4. Message History

Passing every message manually is not practical. LangChain can **store and inject the history automatically**.

**Key components:**
- `ChatMessageHistory` – an in-memory object that stores the messages of one conversation.
- `BaseChatMessageHistory` – the base class (used here as the return type hint).
- `RunnableWithMessageHistory` – wraps a model/chain; before each call it **loads** the past messages, and after each call it **saves** the new user message and AI reply.

**How sessions work:**
- `store` is a dictionary: `{session_id: ChatMessageHistory}`.
- `get_session_history(session_id)` returns the history for that session, creating a new one if it doesn't exist yet.
- Each **session_id** is a separate conversation, so different users/chats don't mix their memory.

In [16]:
from langchain_community.chat_message_histories import ChatMessageHistory  # stores messages in memory
from langchain_core.chat_history import BaseChatMessageHistory              # base class for any history store
from langchain_core.runnables.history import RunnableWithMessageHistory     # adds memory to a runnable

# Dictionary that holds one chat history per session id
store = {}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    # Create a new, empty history the first time we see this session id
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    # Return the existing history for this session
    return store[session_id]

# Wrap the model so it automatically reads/writes the chat history
with_message_history = RunnableWithMessageHistory(
    model,
    get_session_history
)

C:\Users\yokes\AppData\Local\Temp\ipykernel_23196\3539354089.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory


### Config with a `session_id`

When we call the wrapped model, we pass a `config` that tells it **which session** (conversation) to use.

In [17]:
# Use the conversation stored under session id "chat1"
config = {"configurable":{"session_id":"chat1"}}

In [18]:
# 1st message in session "chat1" — this gets saved in the history
response = with_message_history.invoke([
    HumanMessage(
        content = " Hi my name is yoki and I am an AI engineer"
    )
],
config = config
)

C:\Users\yokes\AppData\Local\Temp\ipykernel_23196\3539354089.py:8: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  store[session_id] = ChatMessageHistory()


In [20]:
# 2nd message in the SAME session — we don't resend the earlier message,
# the history is loaded automatically, so the model knows our role
response = with_message_history.invoke([
    HumanMessage(
        content = "what is my role"
    )
],
config = config
)

In [22]:
# Print only the text of the model's reply
print(response.content)

You’re an **AI Engineer**—a role that sits at the intersection of software engineering, data science, and machine learning. Here’s a quick snapshot of what that typically entails:

| Core Responsibility | What It Looks Like in Practice | Key Skills |
|---------------------|--------------------------------|------------|
| **Model Development & Training** | Selecting architectures, fine‑tuning pre‑trained models, experimenting with hyper‑parameters. | PyTorch/TensorFlow, scikit‑learn, Hugging Face 🤗, hyper‑parameter tuning tools. |
| **Data Engineering** | Building pipelines to ingest, clean, transform, and label data at scale. | Spark/Databricks, Airflow, SQL, data versioning (DVC, Delta Lake). |
| **Deployment & Ops** | Packaging models as APIs, scaling inference, monitoring latency & accuracy. | Docker, Kubernetes, TensorFlow Serving / TorchServe, MLflow, Prometheus. |
| **Model Governance** | Ensuring reproducibility, version control, audit trails, and compliance. | Git, DVC, MLflow,

### Changing the session

If we use a **different `session_id`**, we get a brand-new empty history — the model will **not** remember anything from `chat1`.

In [21]:
# A new session id -> new, separate conversation memory
# (invoking with config=config1 and asking "what is my name?" would NOT know the answer)
config1 = {"configurable":{"session_id":"chat2"}}

## 5. Prompt Templates

Prompt templates turn raw user input into a well-structured prompt for the LLM.

- `ChatPromptTemplate.from_messages([...])` – builds a prompt from a list of messages.
- `("system", "...")` – a system message that gives the assistant instructions.
- `MessagesPlaceholder(variable_name="messages")` – a slot where a **list of messages** (our chat) will be inserted at runtime.

We then create a **chain** using the pipe operator `|` (LCEL):
`prompt | model` → the formatted prompt output is passed straight into the model.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

# Build a prompt: a system instruction + a placeholder for the chat messages
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","Your are a helpful assistant. Answer all the question to the nest of your ability"),
        MessagesPlaceholder(
            variable_name="messages"   # the key we must pass when invoking the chain
        )
    ]
)

# Chain: prompt -> model (output of prompt becomes input of model)
chain = prompt | model

In [23]:
# (Already imported above — re-imported here for clarity)
from langchain_core.messages import HumanMessage

In [24]:
# Invoke the chain with a dict; the "messages" key fills the MessagesPlaceholder
chain.invoke(
    {
        "messages":[HumanMessage(content="Hi I am yoki")]
    }
)

AIMessage(content='Hello, Yoki! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi I am yoki". They want a response. Probably just greet them. The system says we are ChatGPT, but developer says answer all questions. There\'s no question. Just greet.'}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 94, 'total_tokens': 159, 'completion_time': 0.07139602, 'completion_tokens_details': {'reasoning_tokens': 42}, 'prompt_time': 0.036468247, 'prompt_tokens_details': None, 'queue_time': 0.37978724, 'total_time': 0.107864267}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_816fc87d18', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a217f8-4650-7290-b07a-2a7dd7bac955-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 65, 'total_tokens': 159, 'output_token_details': {'reasoning': 42}})

### Prompt Template + Message History

Now we wrap the **chain** (instead of the bare model) with `RunnableWithMessageHistory`,
so we get both the system instructions from the prompt **and** automatic memory.

In [25]:
# Add memory to the prompt | model chain
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

In [26]:
# New session "chat3"
config = {
    "configurable":{
        "session_id":"chat3"
    }
}

# Here the input is just a list of messages, so it maps to the "messages" placeholder
response =  with_message_history.invoke(
    [HumanMessage(content="Hi I am yoki")],
    config=config
)

In [27]:
# Show the full AIMessage object (content + metadata)
response

AIMessage(content='Hello, Yoki! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'User says: "Hi I am yoki". Probably greeting. We respond politely, ask how can help.'}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 94, 'total_tokens': 139, 'completion_time': 0.045545518, 'completion_tokens_details': {'reasoning_tokens': 22}, 'prompt_time': 0.005271102, 'prompt_tokens_details': None, 'queue_time': 0.376147786, 'total_time': 0.05081662}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_ce54c53bd5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a217fa-28d1-76a4-84a9-5b55ef22a096-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 45, 'total_tokens': 139, 'output_token_details': {'reasoning': 22}})

## 6. Adding More Input Variables to the Prompt

Prompts can contain **extra variables** using `{curly_braces}`.
Here we add a `{language}` variable so the assistant replies in whatever language we choose.

Now the chain expects **two keys** when invoked: `messages` and `language`.

In [31]:
# Adding more complexity: a {language} variable in the system message
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","you are a helpful assistant. Answer all questions to the best of your abilty in {language}"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

# Rebuild the chain with the new prompt
chain = prompt | model

In [32]:
# Pass both variables: the chat messages and the reply language
response = chain.invoke(
    {
        "messages":[HumanMessage(content="Hi I am yoki")],
        "language": "German"
    }
)

response

AIMessage(content='Hallo Yoki! Wie kann ich dir heute helfen?', additional_kwargs={'reasoning_content': 'User says "Hi I am yoki". They want answer in German. Probably greet back.'}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 96, 'total_tokens': 135, 'completion_time': 0.044923629, 'completion_tokens_details': {'reasoning_tokens': 19}, 'prompt_time': 0.004531699, 'prompt_tokens_details': None, 'queue_time': 0.156766181, 'total_time': 0.049455328}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_a12402de73', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a21800-08ab-7529-bdfe-d83edd60d82f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 39, 'total_tokens': 135, 'output_token_details': {'reasoning': 19}})

In [33]:
# Just the reply text (in German)
response.content

'Hallo Yoki! Wie kann ich dir heute helfen?'

## 7. Message History with Multiple Input Keys

Let's now wrap this more complicated chain in a Message History class.
This time, because there are **multiple keys** in the input (`messages` and `language`),
we need to tell `RunnableWithMessageHistory` **which key holds the chat messages** to save in the history.

👉 We do this with `input_messages_key="messages"`.

In [34]:
# Wrap the chain with memory, telling it which input key contains the messages
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"   # only this key is stored in the chat history
)

In [35]:
# New session "chat4"
config = {
    "configurable":
    {
        "session_id":"chat4"
    }
}

# 1st message: introduce ourselves, reply in German
response = with_message_history.invoke(
    {
        "messages":[HumanMessage(content="Hi I am yoki")],
        "language": "German"
    },
    config=config
)

response.content

'Hallo Yoki! Schön, dich kennenzulernen. Wie kann ich dir heute weiterhelfen?'

In [38]:
# 2nd message in the same session — the model should remember our name
response = with_message_history.invoke(
    {
        "messages":[HumanMessage(content="what is my name")],
        "language": "German"
    },
    config=config
)

In [39]:
# "Dein Name ist Yoki." -> "Your name is Yoki." ✅ memory works with multiple input keys
response.content

'Dein Name ist Yoki.'

## 8. Managing the Conversation History (Trimming)

Every message we store is sent to the model again on each call. If the conversation keeps growing:
- it can **exceed the model's context window** (the maximum number of tokens it can read), and
- every call gets **slower and more expensive**.

So we need to **limit how much history we send**. LangChain gives us `trim_messages` to cut the list down to a token budget.

**`trim_messages` parameters used here:**

| Parameter | Value | Meaning |
|---|---|---|
| `max_tokens` | `45` | Maximum number of tokens to keep in the history |
| `strategy` | `"last"` | Keep the **most recent** messages and drop the oldest ones |
| `token_counter` | `model` | Use the model's tokenizer to count tokens |
| `include_system` | `True` | Always keep the `SystemMessage` at the start |
| `allow_partial` | `False` | Never cut a message in half — keep or drop whole messages |
| `start_on` | `"human"` | The kept history must start with a `HumanMessage` (after the system message) |

In [50]:
from langchain_core.messages import SystemMessage, trim_messages

# Build a trimmer that keeps only the latest messages that fit in 45 tokens
trimmer = trim_messages(
    max_tokens=45,          # token budget for the whole history
    strategy="last",        # keep the newest messages, drop the oldest
    token_counter=model,    # count tokens with our Groq model's tokenizer
    include_system=True,    # always keep the system message
    allow_partial=False,    # don't split a message — keep it whole or drop it
    start_on="human"        # history should start with a user message
)

### Example conversation

A sample chat history to test the trimmer: one system message followed by several user/assistant turns.

In [51]:
# A sample conversation (system message + 5 user/assistant turns)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),            # oldest turn — first to be dropped
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!")                       # newest turn
]

In [52]:
# Run the trimmer on its own to see which messages are kept
trimmer.invoke(messages)
# Result: the SystemMessage is kept, the oldest turn ("hi! I'm bob" / "hi!") is dropped,
# and the most recent messages that fit in 45 tokens remain

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

### Adding the trimmer to the chain

We want the trimmer to run **before** the prompt, so only the trimmed history reaches the model.

- `itemgetter("messages")` – takes the `"messages"` value out of the input dictionary.
- `itemgetter("messages") | trimmer` – passes that list through the trimmer.
- `RunnablePassthrough.assign(messages=...)` – keeps all the other input keys (like `language`) unchanged and **replaces** `messages` with the trimmed version.

Flow: **input dict → trim `messages` → prompt → model**

In [55]:
from operator import itemgetter                       # picks a key out of a dict
from langchain_core.runnables import RunnablePassthrough

# Chain: trim the "messages" key, keep "language" as is, then prompt -> model
chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer)
    | prompt
    | model
)

In [56]:
# Ask about something that is still inside the trimmed history
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="What math problem do i Asked")],
        "language": "English",
    }
)

response.content
# ✅ The model remembers "whats 2 + 2" because that message survived the trimming.
# ❌ If we asked "what is my name?", it would NOT know — "hi! I'm bob" was trimmed away.

'You asked: **“whats 2 + 2.”**'

### Trimming + Message History

Finally, we wrap the trimming chain with `RunnableWithMessageHistory`.
Now the chatbot **stores the full history** for each session, but **only sends the most recent messages** (within the token budget) to the model on every call.

In [57]:
# Add memory to the trimming chain
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"   # the key that holds the chat messages
)

In [58]:
# New session "chat5"
config = {
    "configurable":
    {
        "session_id":"chat5"
    }
}

In [ ]:
# Send the sample conversation + a question about the oldest message.
# All of it is saved in the "chat5" history, but the trimmer drops "hi! I'm bob" before it reaches the model.
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="what's my name?")],
        "language": "English",
    },
    config=config,
)

response.content  # expected: the model does NOT know the name (it was trimmed)

In [ ]:
# Ask about a recent message in the same session — it is still within the token budget
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did I ask?")],
        "language": "English",
    },
    config=config,
)

response.content  # expected: the model remembers "whats 2 + 2"

## ✅ Summary

| Concept | What it does |
|---|---|
| `load_dotenv()` | Loads API keys from the `.env` file |
| `ChatGroq` | Connects to an LLM hosted on Groq |
| `HumanMessage` / `AIMessage` / `SystemMessage` | Represent user, assistant and system messages |
| `ChatMessageHistory` + `store` | Keep the messages of each session in memory |
| `RunnableWithMessageHistory` | Automatically loads & saves chat history for each call |
| `session_id` in `config` | Selects which conversation to use |
| `ChatPromptTemplate` + `MessagesPlaceholder` | Build structured prompts with a slot for chat messages |
| `prompt \| model` | Chain the prompt and model together (LCEL) |
| `input_messages_key` | Tells the history wrapper which input key holds the messages |
| `trim_messages` | Cuts the history down to a token budget so it fits the context window |
| `RunnablePassthrough.assign` + `itemgetter` | Change one input key (trim `messages`) while passing the others through |

**Key takeaway:** the session history keeps *everything*, but the trimmer decides what the model actually *sees*. A small `max_tokens` saves cost but makes the bot forget older details (like the user's name).